In [0]:
from pyspark.sql.types import StructType, StructField,StringType, IntegerType, DateType, TimestampType, FloatType 

import pyspark.sql.functions as F

In [0]:
catalog_name = 'ecommerce'

###Read and Vakidate Bronze brands table

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_brands");
df_bronze.show(10)

### Silver Clean Data Frame

In [0]:
df_silver = df_bronze.withColumn("brand_name",F.trim(F.col("brand_name")));
df_silver.limit(5).show()

In [0]:
df_silver = df_silver.withColumn("brand_code", F.regexp_replace(F.col("brand_code"),r"[^A-Za-z0-9]",""))
df_silver.limit(10).show()

In [0]:
df_silver.select("category_code").distinct().show()


In [0]:
#Anomalies Dictionary
anomalies ={
    "GROCERY" : "GRCY",
    "BOOKS" : "BKS",
    "TOYS" : "TOY"
}

df_silver = df_silver.replace(anomalies, subset="category_code")


In [0]:
df_silver.select("category_code").distinct().show()

###Drop the table in spark

In [0]:
spark.sql(f"Drop table {catalog_name}.silver.slv_brands")

In [0]:
df_silver.printSchema()

In [0]:
#Writing raw_data into the silver layer (catalog name : ecommerce, schema name: silver, table name : slv_brands)

df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f"{catalog_name}.silver.slv_brands")

#Category

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_category")
df_bronze.show(10)

### Finding Duplicate Records

In [0]:
df_duplicate = df_bronze.groupBy("category_code").count().filter(F.col("count")>1)
display(df_duplicate)

###Clearing Duplicate Records

In [0]:
df_silver =df_bronze.dropDuplicates(["CATEGORY_CODE"])
display(df_silver)

###Coverting to UpperCase

In [0]:
df_silver = df_silver.withColumn("category_code",F.upper(F.col("category_code")))
display(df_silver)

###Create Delta Table - slv_category

In [0]:
df_silver.write.format("delta")\
    .mode("overwrite")\
        .option("mergeSchema", "true")\
            .saveAsTable(f"{catalog_name}.silver.slv_category");

#Prducts

In [0]:
#Read the raw data from the bronze table (ecommerce.bronze.brz_products)
df_bronze = spark.table(f"{catalog_name}.bronze.brz_products")
display(df_bronze)

###Counting Rows and Columns

In [0]:
row_count, column_count = df_bronze.count(), len(df_bronze.columns)

print(f"Rount Count is :{row_count}")
print(f"Column Count is : {column_count}")